<center>
    <img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/Logos/organization_logo/organization_logo.png" width="300" alt="cognitiveclass.ai logo">
</center>

#### Import the required libraries we need for the lab.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pyplot
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

#### Read the dataset in the csv file from the URL

In [ ]:
boston_df=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv')

# Boston Housing: Statistics for Data Science Final Assignment

In this notebook I analyze the Boston Housing dataset. The notebook has three parts:

1. Become familiar with the dataset
2. Generate descriptive statistics and visualizations
3. Use appropriate statistical tests to answer four questions

**Variables used in this notebook**

- **CHAS**: Charles River dummy variable (1 if the tract bounds the river, 0 otherwise)
- **NOX**: Nitric oxides concentration (parts per 10 million)
- **INDUS**: Proportion of non-retail business acres per town
- **AGE**: Proportion of owner-occupied units built prior to 1940
- **DIS**: Weighted distances to five Boston employment centres
- **PTRATIO**: Pupil-teacher ratio by town
- **MEDV**: Median value of owner-occupied homes in $1000's

## Task 1 and 2: Become familiar with the dataset and load it

In [ ]:
# Size of the dataset and the first rows
print(boston_df.shape)
boston_df.head()

In [ ]:
# Data types of all the columns
boston_df.dtypes

In [ ]:
# More information about the columns
boston_df.info()

### Check for missing values in each column

In [ ]:
boston_df.isnull().sum()

### Check for duplicate rows

In [ ]:
print('Number of duplicate rows:', boston_df.duplicated().sum())
boston_df[boston_df.duplicated()]

### Correlation matrix of the dataset

In [ ]:
corr_matrix = boston_df.drop(columns=['Unnamed: 0'], errors='ignore').corr(numeric_only=True)
corr_matrix

In [ ]:
pyplot.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
pyplot.title('Correlation matrix heatmap')
pyplot.show()

**Observation:** The matrix shows how strongly each pair of numeric variables is related. For example, NOX and INDUS have a strong positive correlation, and MEDV is strongly related to variables such as RM and LSTAT.

In [ ]:
# Descriptive statistics for every numeric column
boston_df.describe()

## Task 3: Descriptive statistics and visualizations

### 3.1 Boxplot of the median value of owner-occupied homes (MEDV)

In [ ]:
sns.boxplot(y='MEDV', data=boston_df)
pyplot.title('Boxplot of the median value of owner-occupied homes')
pyplot.ylabel('MEDV (in $1000\'s)')
pyplot.show()

**Observation:** The median value of homes is centered around 21 to 22 thousand dollars. The boxplot shows several outliers on the high end, which are the most expensive tracts.

### 3.2 Bar chart for the Charles River variable (CHAS)

In [ ]:
sns.countplot(x='CHAS', data=boston_df)
pyplot.title('Bar chart of the Charles River variable')
pyplot.xlabel('CHAS (0 = does not bound the river, 1 = bounds the river)')
pyplot.ylabel('Number of tracts')
pyplot.show()

**Observation:** Most tracts do not bound the Charles River. Only a small number of tracts (CHAS = 1) are next to the river.

### 3.3 Boxplot of MEDV vs AGE (discretized into three groups)

In [ ]:
# Discretize AGE into three groups
boston_df.loc[(boston_df['AGE'] <= 35), 'age_group'] = '35 years and younger'
boston_df.loc[(boston_df['AGE'] > 35) & (boston_df['AGE'] < 70), 'age_group'] = 'between 35 and 70 years'
boston_df.loc[(boston_df['AGE'] >= 70), 'age_group'] = '70 years and older'

order = ['35 years and younger', 'between 35 and 70 years', '70 years and older']
sns.boxplot(x='age_group', y='MEDV', data=boston_df, order=order)
pyplot.title('MEDV by age group of the housing units')
pyplot.xlabel('Age group')
pyplot.ylabel('MEDV (in $1000\'s)')
pyplot.show()

**Observation:** Median home values are highest for the group with the newest housing units (35 years and younger) and lowest for the group with the oldest units (70 years and older). The group in the middle falls in between.

### 3.4 Scatter plot of NOX vs INDUS

In [ ]:
sns.scatterplot(x='NOX', y='INDUS', data=boston_df)
pyplot.title('Nitric oxide concentration vs proportion of non-retail business acres')
pyplot.xlabel('NOX (nitric oxides concentration)')
pyplot.ylabel('INDUS (proportion of non-retail business acres per town)')
pyplot.show()

**Observation:** There is a positive relationship between the two variables. Towns with a larger proportion of non-retail business acres tend to have higher nitric oxide concentrations.

### 3.5 Histogram for the pupil to teacher ratio (PTRATIO)

In [ ]:
pyplot.hist(boston_df['PTRATIO'], bins=15, edgecolor='black')
pyplot.title('Histogram of the pupil to teacher ratio')
pyplot.xlabel('PTRATIO')
pyplot.ylabel('Number of tracts')
pyplot.show()

**Observation:** The pupil to teacher ratio is not symmetric. Many tracts have a ratio around 20 to 21, and the ratio is lower in a smaller number of tracts.

## Task 4: Hypothesis tests

All tests use a significance level of **alpha = 0.05**.

### 4.1 Levene's test: are the variances of MEDV equal for houses bounded by the Charles River (CHAS = 1) and houses that are not (CHAS = 0)?

**Test:** Levene's test for equality of variances

- **H0:** The variances of MEDV are equal in the two groups.
- **H1:** The variances of MEDV are not equal in the two groups.

In [ ]:
chas_yes = boston_df[boston_df['CHAS'] == 1]['MEDV']
chas_no = boston_df[boston_df['CHAS'] == 0]['MEDV']

levene_stat, levene_p = scipy.stats.levene(chas_yes, chas_no, center='mean')
print('Levene test statistic: {}'.format(levene_stat))
print('Levene p-value: {}'.format(levene_p))

# Decide which version of the t-test to use in the next step
equal_var = levene_p > 0.05
print('Assume equal variances:', equal_var)

**Interpretation:** We compare the p-value of Levene's test with alpha = 0.05. If the p-value is below 0.05, we reject H0, the variances are not equal, and the next t-test uses the unequal variance version (Welch's t-test). If the p-value is above 0.05, we do not reject H0 and the t-test assumes equal variances. The code above makes this choice with the `equal_var` variable.

### 4.2 Is there a significant difference in the median value of houses bounded by the Charles River or not?

**Test:** T-test for independent samples

- **H0:** There is no difference in the mean MEDV between tracts that bound the Charles River and tracts that do not (mu1 = mu2).
- **H1:** There is a difference in the mean MEDV between the two groups (mu1 != mu2).

In [ ]:
t_stat, t_p = scipy.stats.ttest_ind(chas_yes, chas_no, equal_var=equal_var)
print('T-statistic: {}'.format(t_stat))
print('p-value: {}'.format(t_p))

**Conclusion:** The p-value of the t-test is much smaller than 0.05, so we reject the null hypothesis. There is a statistically significant difference in the median value of houses bounded by the Charles River and houses that are not. Houses next to the river have a higher mean MEDV.

### 4.3 Is there a difference in the median values of houses (MEDV) for each proportion of owner-occupied units built prior to 1940 (AGE)?

**Test:** One-way ANOVA (three age groups)

- **H0:** The mean MEDV is the same in all three age groups (mu1 = mu2 = mu3).
- **H1:** At least one age group has a different mean MEDV.

In [ ]:
young = boston_df[boston_df['age_group'] == '35 years and younger']['MEDV']
middle = boston_df[boston_df['age_group'] == 'between 35 and 70 years']['MEDV']
old = boston_df[boston_df['age_group'] == '70 years and older']['MEDV']

# Levene's test for equality of variances across the three groups
print("Levene's test:", scipy.stats.levene(young, middle, old, center='mean'))

f_statistic, p_value = scipy.stats.f_oneway(young, middle, old)
print('F-statistic: {}, p-value: {}'.format(f_statistic, p_value))

**Conclusion:** The p-value of the ANOVA test is far below 0.05, so we reject the null hypothesis. There is a statistically significant difference in the median value of houses across the three age groups.

### 4.4 Can we conclude that there is no relationship between nitric oxide concentrations and the proportion of non-retail business acres per town?

**Test:** Pearson correlation

- **H0:** NOX and INDUS are not correlated (the correlation is 0).
- **H1:** NOX and INDUS are correlated (the correlation is not 0).

In [ ]:
corr, p_value = scipy.stats.pearsonr(boston_df['NOX'], boston_df['INDUS'])
print('Pearson correlation: {}, p-value: {}'.format(corr, p_value))

**Conclusion:** The p-value is far below 0.05, so we reject the null hypothesis. We cannot conclude that there is no relationship. There is a statistically significant and strong positive correlation between nitric oxide concentrations and the proportion of non-retail business acres per town, which agrees with the scatter plot.

### 4.5 What is the impact of an additional weighted distance to the five Boston employment centres on the median value of owner-occupied homes?

**Test:** Simple linear regression (MEDV as the response, DIS as the predictor)

- **H0:** The slope of DIS is 0 (DIS has no effect on MEDV).
- **H1:** The slope of DIS is not 0.

In [ ]:
X = sm.add_constant(boston_df['DIS'])
y = boston_df['MEDV']
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
print('Intercept:', model.params['const'])
print('Slope for DIS:', model.params['DIS'])
print('R-squared:', model.rsquared)
print('p-value for DIS:', model.pvalues['DIS'])

**Conclusion:** The p-value of the DIS coefficient is below 0.05, so we reject the null hypothesis. The weighted distance to the five Boston employment centres has a statistically significant positive effect on the median value of homes: for each additional unit of weighted distance, the median home value increases on average by the value of the DIS slope shown above (in $1000's). The R-squared is small, so distance explains only a small part of the variation in home values, and other variables matter as well.